# Tolerance Range Selection
## Context
To calculate the travel time between each stop for every bus trip, we perform a complex asof_join, which requires a manual setting of `tolerance` range to work well.
## Goal
Find the ideal tolerance for each stop pair for each route.

## (below for future documentation)
## Find ideal tolerance in asof_join
1. Required data: `bus_event_time.parquet`, `target_stop.json`
2. Create ideal_tolerances = []
2. For each route, set tolerance = '1h'
3. Perform asof_join, get the number of rows of resulting joined df (`nrows: int`)
3. Repeat the following
    - increase tolerance by '1h' (new_tolerance)
    - perform asof_join again, get new_nrows
    - if new_nrows / nrows < 1.03 or tolerance == "12h", ideal_tolerance = tolerance, break the loop
    - else, tolerance = new_tolerance, continue the loop
6. ideal_tolerances.append({"depart": int, "dest": int, "tolerance": str})
7. Create pl.DataFrame(ideal_tolerances)
8. df.write_csv

# Import

In [ ]:
import polars as pl
from constants import DATA_FILE, PROCESSED_DATA_FOLDER
from helpers import clean_df
import json
import plotly.express as px
from datetime import datetime, date, time

pl.Config.set_tbl_rows(30)

polars.config.Config

In [44]:
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    target_routes: list[str] = json.load(f)
df_train = (
    pl.scan_parquet(DATA_FILE)
    .filter(pl.col("SubRouteID").is_in(target_routes))
    .pipe(clean_df)
)
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "r", encoding="utf-8") as f:
    target_stops: dict[str, list[int]] = json.load(f)
with open(PROCESSED_DATA_FOLDER / "stops_dict.json", "r", encoding="utf-8") as f:
    stops_dict: dict[str, str] = json.load(f)

# Find tolerance ranges through asif_join

In [69]:
def asof_join_by_id(
    route_df: pl.DataFrame,
    depart_id: int,
    dest_id: int,
    tolerance: str,
) -> int:
    depart_df = (
        route_df.filter(
            pl.col("Event") == "離站",
            pl.col("StopID") == depart_id,
        )
        .with_columns(pl.col("Time").alias("DepartureTime"))
        .sort("Time")
    )
    dest_df = (
        route_df.filter(
            pl.col("Event") == "進站",
            pl.col("StopID") == dest_id,
        )
        .with_columns(pl.col("Time").alias("ArrivalTime"))
        .sort("Time")
    )
    result = depart_df.join_asof(
        dest_df,
        on="Time",
        by="Plate",
        strategy="forward",
        tolerance=tolerance,
        check_sortedness=False,
    ).drop_nulls(subset=["ArrivalTime"])
    return len(result)

In [70]:
def find_ideal_tolerances(
    df: pl.LazyFrame,
    target_stops: dict[str, list[int]],
) -> pl.DataFrame:
    """
    Finds the ideal asof_join tolerance for each sequential stop pair per route.

    Args:
        df: Cleaned LazyFrame (output of clean_df).
        target_stops: Maps Route (str) to an ordered list of StopIDs (int).
                      Pairs are derived from consecutive stops in the list.

    Returns:
        A DataFrame with columns: route, depart_id, dest_id, tolerance.
    """

    def _find_tolerance_for_pair(
        route_df: pl.DataFrame,
        depart_id: int,
        dest_id: int,
    ) -> tuple[str, int]:
        MAX_HOURS = 12
        tolerance_hours = 1
        n_rows = asof_join_by_id(route_df, depart_id, dest_id, f"{tolerance_hours}h")

        while tolerance_hours < MAX_HOURS:
            new_tolerance_hours = tolerance_hours + 1
            new_nrows = asof_join_by_id(
                route_df, depart_id, dest_id, f"{new_tolerance_hours}h"
            )
            if n_rows > 0 and new_nrows / n_rows < 1.03:
                break
            tolerance_hours = new_tolerance_hours
            n_rows = new_nrows

        return (f"{tolerance_hours}h", n_rows)

    collected_df = df.select(["Route", "Plate", "StopID", "Event", "Time"]).collect(
        engine="streaming"
    )
    results: list[dict] = []
    count = 0
    for route, stop_ids in target_stops.items():
        route_df = collected_df.filter(pl.col("Route") == route)
        if route_df.is_empty():
            continue

        for depart_id, dest_id in zip(stop_ids, stop_ids[1:]):
            tolerance, n_rows = _find_tolerance_for_pair(route_df, depart_id, dest_id)
            results.append(
                {
                    "route": route,
                    "depart_id": depart_id,
                    "dest_id": dest_id,
                    "tolerance": tolerance,
                    "n_rows": n_rows,
                }
            )
        count += 1
        if count % 100 == 0:
            print(f"{count} routes have been processed.")

    return pl.DataFrame(
        results,
        schema={
            "route": pl.String,
            "depart_id": pl.Int32,
            "dest_id": pl.Int32,
            "tolerance": pl.String,
            "n_rows": pl.UInt16,
        },
    )

In [6]:
ideal_tolerances = find_ideal_tolerances(df_train, target_stops)

100 routes have been processed.
200 routes have been processed.
300 routes have been processed.
400 routes have been processed.
500 routes have been processed.
600 routes have been processed.
700 routes have been processed.
800 routes have been processed.
900 routes have been processed.
1000 routes have been processed.
1100 routes have been processed.
1200 routes have been processed.
1300 routes have been processed.
1400 routes have been processed.
1500 routes have been processed.
1600 routes have been processed.


- It took 9 minutes 49 seconds to finish 1666 routes.

In [8]:
ideal_tolerances.write_csv(PROCESSED_DATA_FOLDER / "ideal_tolerances.csv")

# Investigate anomalies in ideal_tolerances (12h)
- Load ideal_tolerances without performing the search again

In [49]:
ideal_tolerances = pl.read_csv(PROCESSED_DATA_FOLDER / "ideal_tolerances.csv")

In [ ]:
hour_order = ["1h", "2h", "3h", "4h", "5h", "6h", "12h"]

fig = px.histogram(
    ideal_tolerances.sort("tolerance", descending=True),
    x="tolerance",
    category_orders={"tolerance": hour_order},
    color="tolerance",
    color_discrete_sequence=px.colors.sequential.Teal,
    labels={"tolerance": "Tolerance window", "count": "Count"},
    title="Distribution of ideal tolerances",
)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter, sans-serif", size=13, color="#444"),
    title=dict(
        font=dict(size=16, color="#222"), x=0, xanchor="left", pad=dict(l=0, b=16)
    ),
    xaxis=dict(
        categoryorder="array",
        categoryarray=hour_order,
        showgrid=False,
        showline=True,
        linecolor="#ddd",
        tickfont=dict(size=12),
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#f0f0f0",
        showline=False,
        tickfont=dict(size=12),
    ),
    bargap=0.25,
    margin=dict(l=40, r=20, t=60, b=40),
)

fig.update_traces(marker_line_width=0)

fig.show()

In [ ]:
fig = px.histogram(
    ideal_tolerances,
    x="n_rows",
    labels={"n_rows": "Number of Rows", "count": "Count"},
    color_discrete_sequence=["#1FC2B0"],
    title="Distribution of row counts",
)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter, sans-serif", size=13, color="#444"),
    title=dict(
        font=dict(size=16, color="#222"), x=0, xanchor="left", pad=dict(l=0, b=16)
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor="#ddd",
        tickfont=dict(size=12),
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#f0f0f0",
        showline=False,
        tickfont=dict(size=12),
    ),
    bargap=0.25,
    margin=dict(l=40, r=20, t=60, b=40),
)

fig.update_traces(marker_line_width=0)

fig.show()

## Routes with 12h tolerances
- By the previous stops selection algorithm, there shouldn't be any stop pairs that require 12h tolerance ranges

In [ ]:
ideal_tolerances.filter(pl.col("tolerance") == "12h").select(
    pl.col("n_rows").sum().alias("total_rows_matched_for_12h")
)

total_rows_matched_for_12h
i64
0


- All the routes with 12h tolerance have **zero** rows matched, suggesting that rather than the routes need 12h tolerance, there are data issues present for these routes.

In [73]:
abnormal_routes = ideal_tolerances.filter(pl.col("tolerance") == "12h").select(
    ["route", "depart_id", "dest_id"]
)
print(f"There are {len(abnormal_routes)} abnormal routes with 12h as their tolerances.")

There are 14 abnormal routes with 12h as their tolerances.


In [75]:
ideal_tolerances.filter(pl.col("tolerance") == "12h").select(
    ["route", "depart_id", "dest_id"]
).sort("route")

route,depart_id,dest_id
str,i64,i64
"""1613A2""",266866,300440
"""1633B2""",266216,300702
"""1633C2""",266216,300702
"""183701""",268417,311648
"""183702""",311648,268417
"""183801""",268417,268482
"""183802""",268481,268417
"""186201""",268417,268482
"""186202""",268481,268417


# Abnormal Routes


## Investigate: 1613A2

In [84]:
df_train.filter(
    pl.col("Route") == "1613A2", pl.col("StopID").is_in([266866, 300440])
).collect()

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""1613A2""","""FAB-312""","""返程""","""修德國小""",300440,5,"""進站""",2025-12-12 19:31:56,"""Fri"""
"""1613A2""","""FAB-312""","""返程""","""修德國小""",300440,5,"""離站""",2025-12-12 19:32:08,"""Fri"""
"""1613A2""","""FAB-312""","""返程""","""仁德""",266866,2,"""進站""",2026-01-09 13:05:56,"""Fri"""
"""1613A2""","""FAB-312""","""返程""","""仁德""",266866,2,"""離站""",2026-01-09 13:07:10,"""Fri"""


- Insufficient data. **Abandon** route **1613A2**

## Investigate: 183701

In [87]:
abnormal_df = df_train.filter(
    pl.col("Route") == "183701", pl.col("StopID").is_in([268417, 311648])
).collect()

In [17]:
d = date(2025, 11, 6)
abnormal_df.filter(pl.col("Time").is_between(d, datetime.combine(d, time(23, 59, 59))))

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-1167""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-11-06 14:07:42,"""Thu"""
"""183701""","""KKA-1167""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-11-06 14:10:19,"""Thu"""
"""183701""","""KKA-2611""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-11-06 18:20:25,"""Thu"""
"""183701""","""KKA-2611""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-11-06 18:23:41,"""Thu"""


In [18]:
d = date(2025, 3, 1)
abnormal_df.filter(pl.col("Time").is_between(d, datetime.combine(d, time(23, 59, 59))))

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-1125""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-01 10:21:23,"""Sat"""
"""183701""","""KKA-1125""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-01 10:22:50,"""Sat"""
"""183701""","""KKA-1107""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-01 12:22:06,"""Sat"""
"""183701""","""KKA-1107""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-01 12:25:05,"""Sat"""
"""183701""","""KKA-1103""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-01 16:21:21,"""Sat"""
"""183701""","""KKA-1103""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-01 16:23:27,"""Sat"""
"""183701""","""KKA-1109""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-01 20:22:35,"""Sat"""
"""183701""","""KKA-1109""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-01 20:32:28,"""Sat"""


### Finding
- For unknown reasons, on any single days, only *one* of the two stops (朝馬站國光, 永康轉運站) was recorded.
- Solution: Since in later deployment, when the user wants to predict bus travel time of a specific route, the model would make predictions sequentially and add up all the predicted times at the end, data issues like this would render the prediction of the entire route null. Therefore, **abandon** 183701, 183702

## Ivestigate: 183801

In [105]:
d = date(2025, 3, 2)
df_train.filter(
    pl.col("Route") == "183801",
    pl.col("StopID").is_in([268417, 268482]),
    pl.col("Time").is_between(d, datetime.combine(d, time(23, 59, 59))),
).collect()

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183801""","""FAB-939""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-02 11:22:28,"""Sun"""
"""183801""","""FAB-939""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-02 11:33:49,"""Sun"""
"""183801""","""071-U6""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-02 15:33:21,"""Sun"""
"""183801""","""071-U6""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-02 15:41:39,"""Sun"""
"""183801""","""768-FY""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-02 18:10:10,"""Sun"""
"""183801""","""768-FY""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-02 18:16:06,"""Sun"""
"""183801""","""499-FS""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-02 19:42:32,"""Sun"""
"""183801""","""499-FS""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-02 19:53:24,"""Sun"""
"""183801""","""FAB-911""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-02 21:39:55,"""Sun"""


- This route shares the same problem with route 183701.
- **Abandon route 183801, 183802**

## Investigate: 187201

In [121]:
df_train.filter(
    pl.col("Route") == "187201",
    pl.col("StopID").is_in([268417, 268482]),
    pl.col("Time").is_between(date(2025, 3, 1), datetime(2025, 3, 1, 23, 59, 59)),
).collect()

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""187201""","""768-FY""","""去程""","""朝馬站(國光)""",268417,3,"""進站""",2025-03-01 06:00:56,"""Sat"""
"""187201""","""768-FY""","""去程""","""朝馬站(國光)""",268417,3,"""離站""",2025-03-01 06:03:37,"""Sat"""
"""187201""","""069-U6""","""去程""","""朝馬站(國光)""",268417,3,"""進站""",2025-03-01 07:02:56,"""Sat"""
"""187201""","""069-U6""","""去程""","""朝馬站(國光)""",268417,3,"""離站""",2025-03-01 07:06:44,"""Sat"""
"""187201""","""FAB-088""","""去程""","""朝馬站(國光)""",268417,3,"""進站""",2025-03-01 08:59:11,"""Sat"""
"""187201""","""FAB-088""","""去程""","""朝馬站(國光)""",268417,3,"""離站""",2025-03-01 09:03:39,"""Sat"""
"""187201""","""965-U7""","""去程""","""朝馬站(國光)""",268417,3,"""進站""",2025-03-01 10:05:32,"""Sat"""
"""187201""","""965-U7""","""去程""","""朝馬站(國光)""",268417,3,"""離站""",2025-03-01 10:11:12,"""Sat"""
"""187201""","""881-FW""","""去程""","""朝馬站(國光)""",268417,3,"""進站""",2025-03-01 12:05:36,"""Sat"""


### Finding
- After more searching about these routes, it turns out the problem lie in the fact that these routes changed operators last year, leading to slight variation in the bus route. Therefore, the data for these routes are not salvagable as they have essentially changed the route (even just slightly) but kept the SubRouteID.
- **Abandon route 187201**

## Investigate: 9117B1

In [14]:
abnormal_df = df_train.filter(
    pl.col("Route") == "9117B1", pl.col("StopID").is_in([309008, 266361])
).collect()

In [16]:
abnormal_df.filter(
    pl.col("Time").is_between(date(2025, 10, 1), datetime(2025, 10, 1, 23, 59, 59))
)

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""9117B1""","""KKB-8827""","""去程""","""林園麥當勞""",266361,5,"""進站""",2025-10-01 10:49:11,"""Wed"""
"""9117B1""","""KKB-8827""","""去程""","""林園麥當勞""",266361,5,"""離站""",2025-10-01 10:49:59,"""Wed"""
"""9117B1""","""KKB-8827""","""去程""","""東港轉運站""",309008,7,"""進站""",2025-10-01 11:15:08,"""Wed"""
"""9117B1""","""KKB-8827""","""去程""","""東港轉運站""",309008,7,"""離站""",2025-10-01 11:21:16,"""Wed"""
"""9117B1""","""KKB-8128""","""去程""","""林園麥當勞""",266361,5,"""進站""",2025-10-01 12:13:06,"""Wed"""
…,…,…,…,…,…,…,…,…
"""9117B1""","""KKB-8771""","""去程""","""東港轉運站""",309008,7,"""離站""",2025-10-01 17:40:05,"""Wed"""
"""9117B1""","""KKB-8725""","""去程""","""林園麥當勞""",266361,5,"""進站""",2025-10-01 19:01:16,"""Wed"""
"""9117B1""","""KKB-8725""","""去程""","""林園麥當勞""",266361,5,"""離站""",2025-10-01 19:01:46,"""Wed"""


- Seems normal

In [19]:
for s in target_stops["9117B1"]:
    print(s, stops_dict[str(s)])

266090 自立站
265969 捷運小港站
309008 東港轉運站
266361 林園麥當勞
284651 枋寮站
266093 車城農會
309632 恆春轉運站
309633 南灣
269822 墾丁派出所


In [23]:
df_train.filter(pl.col("Route") == "9117B1").select(
    ["Route", "StopID", "StopSeq"]
).unique(subset=["Route", "StopID"]).sort("StopSeq").collect()

Route,StopID,StopSeq
str,i32,i32
"""9117B1""",266090,1
"""9117B1""",284643,2
"""9117B1""",266264,3
"""9117B1""",309630,4
"""9117B1""",265969,4
"""9117B1""",309008,5
"""9117B1""",266361,5
"""9117B1""",284651,6
"""9117B1""",266093,7


In [54]:
df_train.filter(pl.col("Route") == "9117B1", pl.col("StopID") == 309008).select(
    ["Route", "StopID", "StopSeq"]
).unique(subset=["StopID", "StopSeq"]).sort("StopSeq").collect()

Route,StopID,StopSeq
str,i32,i32
"""9117B1""",309008,5
"""9117B1""",309008,7


In [55]:
df_train.filter(pl.col("Route") == "9117B1", pl.col("StopID") == 266361).select(
    ["Route", "StopID", "StopSeq"]
).unique(subset=["StopID", "StopSeq"]).sort("StopSeq").collect()

Route,StopID,StopSeq
str,i32,i32
"""9117B1""",266361,5


In [60]:
df_train.filter(
    pl.col("Route") == "9117B1", pl.col("StopID") == 309008, pl.col("StopSeq") == 5
).collect().select("Time").describe()

statistic,Time
str,str
"""count""","""12"""
"""null_count""","""0"""
"""mean""","""2025-02-28 15:21:02.500000"""
"""std""",null
"""min""","""2025-02-28 11:10:15"""
"""25%""","""2025-02-28 11:58:00"""
"""50%""","""2025-02-28 16:48:41"""
"""75%""","""2025-02-28 17:36:12"""
"""max""","""2025-02-28 19:23:31"""


- The problem is due to faulty Stop Sequence is due to faulty StopSequence in `stops_dict`, whose error originates from wrong StopSeq in the original dataset. And the error only happened on 2025-02-28. Stop 309008 should be the 7th stop of the route but one that day it was assigned the 5th stop, which messes up the asof_join. 
- After manual inspection of the route, it is discovered that **removing Stop 266361** in **9117B1** would solve the problem entirely as the following shows:

In [ ]:
new_stops = [s for s in target_stops["9117B1"] if s != 266361]
for i in range(len(new_stops) - 1):
    print(
        f"Between Stop {new_stops[i]} and {new_stops[i + 1]}, number of matched rows = {
            asof_join_by_id(
                df_train.filter(pl.col('Route') == '9117B1').collect(),
                new_stops[i],
                new_stops[i + 1],
                '1h',
            )
        }"
    )

Between Stop 266090 and 265969, number of matched rows = 1901
Between Stop 265969 and 309008, number of matched rows = 1946
Between Stop 309008 and 284651, number of matched rows = 1908
Between Stop 284651 and 266093, number of matched rows = 1883
Between Stop 266093 and 309632, number of matched rows = 1897
Between Stop 309632 and 309633, number of matched rows = 1876
Between Stop 309633 and 269822, number of matched rows = 1841


In [82]:
for s in target_stops["9117B2"]:
    print(s, stops_dict[str(s)])

266201 小灣
266430 南灣
260803 恆春轉運站
284704 車城農會
261650 枋寮站
309008 東港轉運站
278270 高雄國際機場(國內)
266099 林園麥當勞
269929 建國站


- Here the StopSequence of Stop 266099 is messed up. **Remove 266099** for route **9117B2**

In [83]:
new_stops = [s for s in target_stops["9117B2"] if s != 266099]
for i in range(len(new_stops) - 1):
    print(
        f"Between Stop {new_stops[i]} and {new_stops[i + 1]}, number of matched rows = {
            asof_join_by_id(
                df_train.filter(pl.col('Route') == '9117B2').collect(),
                new_stops[i],
                new_stops[i + 1],
                '1h',
            )
        }"
    )

Between Stop 266201 and 266430, number of matched rows = 1890
Between Stop 266430 and 260803, number of matched rows = 1953
Between Stop 260803 and 284704, number of matched rows = 1923
Between Stop 284704 and 261650, number of matched rows = 1878
Between Stop 261650 and 309008, number of matched rows = 1949
Between Stop 309008 and 278270, number of matched rows = 1805
Between Stop 278270 and 269929, number of matched rows = 1896


## Other routes
- **Abandoning routes 1633B2, 1633C2, 186201, 186202, 750002,** since these have very few trips per week, so it's not worth the effort the investigate and potentially save the data

# Descrepancy between `target_routes` and `target_stops`
- I discovered that one route that is in target_routes but not in target_stops, which shouldn't happen

In [28]:
print(
    f"The difference: {sorted([s for s in target_routes if s not in target_stops.keys()])}"
)

The difference: ['113302']


In [29]:
df_train.filter(pl.col("Route") == "113302").collect()

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat


- There's no data for this route, **abandon route 113302**

# Update `target_routes` and `target_stops`

In [35]:
abandoned_routes = [
    str(s)
    for s in [
        113302,
        "1633B2",
        "1633C2",
        186201,
        186202,
        750002,
        "1613A2",
        183701,
        183702,
        183801,
        183802,
        187201,
    ]
]
print(f"Abandoning {len(abandoned_routes)} routes")

Abandoning 12 routes


In [ ]:
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    target_routes = json.load(f)

In [41]:
new_routes = [s for s in target_routes if s not in abandoned_routes]
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "w", encoding="utf-8") as f:
    json.dump(new_routes, f)
print(f"Number of routes left: {len(new_routes)}")

Number of routes left: 1644


In [47]:
# For routes 9117B1, 9117B2
target_stops["9117B1"].remove(266361)
target_stops["9117B2"].remove(266099)

In [53]:
# convert to pandas to update specific rows more easily
ideal_tolerances_pd = ideal_tolerances.to_pandas()

In [58]:
ideal_tolerances_pd[ideal_tolerances_pd["route"] == "9117B1"]

,route,depart_id,dest_id,tolerance,n_rows
5228,9117B1,266090,265969,1h,1900
5229,9117B1,265969,309008,1h,1924
5230,9117B1,309008,266361,12h,0
5231,9117B1,266361,284651,2h,1876
5232,9117B1,284651,266093,1h,1867
5233,9117B1,266093,309632,1h,1895
5234,9117B1,309632,309633,1h,1876
5235,9117B1,309633,269822,1h,1841


In [64]:
ideal_tolerances_pd.drop([5230], inplace=True)
ideal_tolerances_pd.loc[5231, "depart_id"] = 309008
ideal_tolerances_pd[ideal_tolerances_pd["route"] == "9117B1"]

,route,depart_id,dest_id,tolerance,n_rows
5228,9117B1,266090,265969,1h,1900
5229,9117B1,265969,309008,1h,1924
5231,9117B1,309008,284651,2h,1876
5232,9117B1,284651,266093,1h,1867
5233,9117B1,266093,309632,1h,1895
5234,9117B1,309632,309633,1h,1876
5235,9117B1,309633,269822,1h,1841


In [66]:
ideal_tolerances_pd[ideal_tolerances_pd["route"] == "9117B2"]

,route,depart_id,dest_id,tolerance,n_rows
2631,9117B2,266201,266430,1h,1890
2632,9117B2,266430,260803,1h,1953
2633,9117B2,260803,284704,1h,1923
2634,9117B2,284704,261650,1h,1878
2635,9117B2,261650,309008,1h,1949
2636,9117B2,309008,278270,2h,1909
2637,9117B2,278270,266099,12h,0
2638,9117B2,266099,269929,2h,1892


In [68]:
ideal_tolerances_pd.drop([2637], inplace=True)
ideal_tolerances_pd.loc[2638, "depart_id"] = 278270
ideal_tolerances_pd[ideal_tolerances_pd["route"] == "9117B2"]

,route,depart_id,dest_id,tolerance,n_rows
2631,9117B2,266201,266430,1h,1890
2632,9117B2,266430,260803,1h,1953
2633,9117B2,260803,284704,1h,1923
2634,9117B2,284704,261650,1h,1878
2635,9117B2,261650,309008,1h,1949
2636,9117B2,309008,278270,2h,1909
2638,9117B2,278270,269929,2h,1892


In [ ]:
ideal_tolerances_pd.to_csv(PROCESSED_DATA_FOLDER / "ideal_tolerances.csv", index=False)